# BB84 Quantum Key Distribution - Simulation Notebook

**University of Ruhuna - Dept. of Computer Engineering**  
MIT Licence - see `LICENSE`

This notebook demonstrates the complete BB84 QKD protocol using the
`bb84-qkd-simulator` library.  It covers:

| Section | What it shows |
|---------|---------------|
| 1 | Environment setup and imports |
| 2 | Single simulation — verbose walkthrough |
| 3 | Ideal channel baseline |
| 4 | Eve intercept-resend at 30 %, 50 %, 100 % |
| 5 | Depolarising channel noise (no Eve) |
| 6 | Eve + noise combined |
| 7 | Multi-scenario QBER bar chart |
| 8 | QBER vs Eve intercept-rate sweep |

---

### Quick start
```bash
pip install -r requirements.txt
jupyter notebook examples/bb84_simulation.ipynb
```

## Section 1 - Setup

In [1]:
# Add the repo root to the Python path
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

os.makedirs('../figures', exist_ok=True)

from bb84_config import SimulationConfig
from bb84_runner import run_simulation, run_comparison, PRESET_SCENARIOS
from bb84_plots  import plot_comparison, plot_qber_vs_intercept_rate

print('Imports OK ✓')

Imports OK ✓


---
## Section 2 - Single Simulation (Verbose)

A step-by-step walkthrough of one complete BB84 run:
1. Quantum Transmission
2. Key Sifting
3. QBER Estimation
4. Key Distillation

In [ ]:
cfg = SimulationConfig(
    n_qubits        = 600,
    seed            = 42,
    sample_fraction = 0.15,
    label           = 'Ideal — verbose walkthrough',
)
result = run_simulation(cfg, verbose=True)

### Accessing result fields programmatically

In [ ]:
r = result
print(f'Transmitted     : {r.n_transmitted} qubits')
print(f'Sifted          : {r.n_sifted} bits  ({r.sifted_key_rate:.1%})')
print(f'Final key       : {r.key_length} bits')
print(f'Key gen. rate   : {r.key_generation_rate:.4f} bits/qubit')
print(f'QBER            : {r.qber_result.qber * 100:.2f} %')
print(f'Security status : {r.qber_result.security_status}')
print(f'Keys match      : {r.keys_match}')
print(f'Runtime         : {r.runtime_seconds:.2f} s')

---
## Section 3 - Ideal Channel Baseline

No noise, no eavesdropper.  
Expected QBER ≈ 0 %, status = **SECURE**.

In [ ]:
ideal = run_simulation(
    SimulationConfig(n_qubits=600, seed=42, label='Ideal'),
    verbose=True,
)

---
## Section 4 - Eve: Intercept-Resend Attack

BB84 theory predicts QBER = 0.25 × p_intercept for an intercept-resend attack.  
At full interception (p = 1.0) the expected QBER is **25 %** → **ABORT**.

### 4a - 30 % interception (borderline detection)

In [ ]:
eve_30 = run_simulation(
    SimulationConfig(
        n_qubits=600, seed=42, label='Eve 30%',
        eve_present=True, eve_intercept_prob=0.30,
    ),
    verbose=True,
)

### 4b - 50 % interception

In [ ]:
eve_50 = run_simulation(
    SimulationConfig(
        n_qubits=600, seed=42, label='Eve 50%',
        eve_present=True, eve_intercept_prob=0.50,
    ),
    verbose=True,
)

### 4c - 100 % interception (full eavesdrop)

In [ ]:
eve_100 = run_simulation(
    SimulationConfig(
        n_qubits=600, seed=42, label='Eve 100%',
        eve_present=True, eve_intercept_prob=1.0,
    ),
    verbose=True,
)

---
## Section 5 - Depolarising Channel Noise (No Eve)

Uniform Pauli errors on every gate with probability p = 0.05.  
This models imperfect quantum hardware without any eavesdropper.

In [ ]:
noisy = run_simulation(
    SimulationConfig(
        n_qubits=600, seed=42, label='Noise p=0.05',
        noise_enabled=True, depolar_prob=0.05,
    ),
    verbose=True,
)

---
## Section 6 - Eve + Channel Noise (Worst Case)

Full intercept-resend attack **and** depolarising noise.
QBER expected well above the 11 % abort threshold.

In [ ]:
eve_noise = run_simulation(
    SimulationConfig(
        n_qubits=600, seed=42, label='Eve+Noise',
        eve_present=True, eve_intercept_prob=1.0,
        noise_enabled=True, depolar_prob=0.05,
    ),
    verbose=True,
)

---
## Section 7 - Multi-Scenario QBER Bar Chart

Run all six preset scenarios silently and visualise the results.

In [ ]:
results = run_comparison(PRESET_SCENARIOS)

In [ ]:
plot_comparison(
    PRESET_SCENARIOS,
    results,
    save_path='../figures/qkd_comparison.png',
    subtitle=(
        'BB84 QKD — QBER Comparison: Ideal, Eavesdropping, '
        'and Depolarising Noise  (n = 600 qubits, seed = 42)'
    ),
)

---
## Section 8 - QBER vs Eve Intercept-Rate Sweep

Sweeps Eve's intercept probability from 0 % to 100 % in 10 steps and
overlays the theoretical prediction QBER = 0.25 × p_intercept.

> **Theory recap:** each intercepted qubit has a 50 % chance of Eve choosing
> the wrong basis, and a 50 % chance of Bob getting the wrong bit in that
> case → expected QBER = 0.5 × 0.5 × p = 0.25p.

In [ ]:
plot_qber_vs_intercept_rate(
    n_qubits        = 600,
    steps           = 10,
    sample_fraction = 0.15,
    save_path       = '../figures/qkd_qber_vs_eve.png',
    subtitle        = (
        'BB84 QKD — Simulated vs Theoretical QBER '
        'as a Function of Eve\'s Intercept Probability'
    ),
)

---
## Summary Table

Print a quick reference of all results produced in this notebook.

In [ ]:
runs = [
    ('Ideal',         ideal),
    ('Eve 30%',       eve_30),
    ('Eve 50%',       eve_50),
    ('Eve 100%',      eve_100),
    ('Noise p=0.05',  noisy),
    ('Eve + Noise',   eve_noise),
]

header = f"  {'Scenario':<18} {'QBER':>7}  {'Key bits':>8}  {'Status'}"
print(header)
print('  ' + '-' * 52)
for name, r in runs:
    print(f"  {name:<18} {r.qber_result.qber * 100:>6.2f}%  "
          f"{r.key_length:>8}  {r.qber_result.security_status}")